## Model Selection->

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

In [5]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Lasso,Ridge, LinearRegression, SGDRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor, AdaBoostRegressor
from xgboost import XGBRegressor

#Evaluation & Metrics
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# for saving 
import pickle

# for loading our data
import joblib

In [4]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
print(os.getcwd())


/Users/mac/Desktop/Desktop_Files/Projects/Zomato Restaurant Rating Prediction/src


In [12]:
## loading the prepared dataset->
X = joblib.load("/Users/mac/Desktop/Desktop_Files/Projects/Zomato Restaurant Rating Prediction/artifacts/prepared_data/X.pkl")
Y = joblib.load("/Users/mac/Desktop/Desktop_Files/Projects/Zomato Restaurant Rating Prediction/artifacts/prepared_data/Y.pkl")

### Selecting best models with best random state value->


In [13]:

# Initialize all the models
LR_model = LinearRegression()
RD_model = Ridge()
Lasso_model = Lasso()
DT_model = DecisionTreeRegressor()
KNR_model = KNeighborsRegressor()
RFR_model = RandomForestRegressor()
SGH_model = SGDRegressor()
Bag_model = BaggingRegressor()
GB_model = GradientBoostingRegressor()
XGB_model = XGBRegressor()
ADA_model= AdaBoostRegressor()


# Create a list of models for iteration
models = [
    (LR_model, 'Linear Regression'),
    (RD_model, 'Ridge'),
    (Lasso_model, 'Lasso'),
    (DT_model, 'Decision Tree'),
    (KNR_model, 'KNeighbors'),
    (RFR_model, 'RandomForest'),
    (SGH_model, 'SGDRegressor'),
    (Bag_model, 'Bagging Regressor'),
    (GB_model, 'GradientBoostingRegressor'),
    (XGB_model, 'XGBRegressor'),
    (ADA_model, 'AdaBoostRegressor')
]

In [18]:
X.columns

Index(['votes', 'rating', 'online_order', 'book_table', 'location',
       'rest_type', 'type', 'city'],
      dtype='object')

In [17]:
# Function to check for best random state and R2 score
def maxr2_score(tec, x, y):
    max_r_score = 0
    final_r_state = 0
    for r_state in range(1, 100):
        train_x, test_x, train_y, test_y = train_test_split(x, y, random_state=r_state, test_size=0.30)
        tec.fit(train_x, train_y)
        pred = tec.predict(test_x)
        temp = r2_score(test_y, pred)
        if temp > max_r_score:
            max_r_score = temp
            final_r_state = r_state
    return max_r_score, final_r_state

### Random State Search:
For each model, finds the best random_state out of several splits (default 20) that maximizes R² score, ensuring you use an optimal, reproducible split.

- Train/Test Split:
Splits data into training and test sets using the “best” random state.

- Training & Prediction:
Fits each model on the training set and makes predictions on the test set.

- Metric Calculation:
For each model, computes:

- R² Score (goodness-of-fit)

- Mean Squared Error (MSE)

- Mean Absolute Error (MAE)

Results Aggregation:
Collects all metrics and “best” random states into a DataFrame and sorts it by R² Score descending for easy model selection.

In [16]:

def maxr2_score(model, X, Y, n_iter=20, test_size=0.3):
    """Finds best random_state out of n_iter for highest R2"""
    best_r2, best_state = -float('inf'), None
    for state in range(n_iter):
        X_train, X_test, y_train, y_test = train_test_split(
            X, Y, test_size=test_size, random_state=state
        )
        try:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            score = r2_score(y_test, y_pred)
            if score > best_r2:
                best_r2, best_state = score, state
        except Exception:
            continue  # skip model if it fails
    return best_r2, best_state

results = []

# Iterate through models and calculate the best random state, R2 score, MSE, and MAE
for model, model_name in models:
    max_r2, best_state = maxr2_score(model, X, Y)
    # Final train-test split and evaluation
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=best_state)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    results.append({
        'Model': model_name,
        'Best R2 Score': max_r2,
        'Best Random State': best_state,
        'MSE': mse,
        'MAE': mae,
    })

results_df = pd.DataFrame(results)
sorted_results_df = results_df.sort_values(by='Best R2 Score', ascending=False).reset_index(drop=True)
print(sorted_results_df)

                        Model  Best R2 Score  Best Random State           MSE  \
0                RandomForest       0.310051                 13  38508.973587   
1           Bagging Regressor       0.251713                 13  42171.375795   
2                XGBRegressor       0.206573                  6  44556.265358   
3   GradientBoostingRegressor       0.116775                 11  48317.828718   
4           AdaBoostRegressor       0.046033                 11  52435.039293   
5           Linear Regression       0.025098                 18  53645.066871   
6                       Ridge       0.025098                 18  53645.090165   
7                SGDRegressor       0.024849                 18  53650.784608   
8                       Lasso       0.024569                 18  53674.184133   
9                  KNeighbors      -0.062180                 13  59612.475893   
10              Decision Tree      -0.246138                  6  69730.698222   

           MAE  
0   151.93

## 30% max? Yuck!
- Well I guess the current features just aren’t very predictive atleast thats what Low R² means !
- May be I should try feature engineering and new data sources instead of dropping columns.

## In summary:->

- A 0.31 R² means 31% of the variation (not accuracy) in the target variable is explained by our model.
- not that 31% of predictions are "correct". 
- For regression, it’s about overall fit, not correct/incorrect predictions as in classification.

### Also that makes sense bcz we've seen the Weak correlation between every feature and our target during correlation analysis itself.

## Whats Next->

maybe I'll check other features that can be predicted with more accuracy (atleast >80 this time).
I will update this notebook in case anyone reading this, btw Hello dear ✋.


# Bye 🫡